# Day 4, Notebook 2: the segment summary, and the count it rests on

Notebook 1 produced one number for the whole file. Nobody runs a business unit on one number.

The real question always sits one level down: which part of Kalpa Retail is working, and how sure can anyone be. That means splitting the file by segment and computing the same honest statistic per group.

A second thing starts lying at that point, and it is not the statistic. It is the size of the group.

## Setup

Same file, same conversion, top of the notebook so this runs cold in a fresh Codespace.

`traceback` is here so a deliberate failure can print itself and let the notebook keep running. It is a teaching convenience and nothing more.

In [ ]:
import csv
import statistics
import traceback

DATA_DIR = "../data"
PROFILED_CSV = f"{DATA_DIR}/C2_W01_D04_profiled_STUDENT.csv"

with open(PROFILED_CSV) as f:
    orders = list(csv.DictReader(f))

for r in orders:
    r["amount"] = int(r["amount"])


def show_failure(fn):
    # Run something expected to raise, print the trace, and keep the notebook alive.
    try:
        fn()
    except Exception:
        traceback.print_exc()


print("orders loaded:", len(orders))
print("segments present:", sorted({r["segment"] for r in orders}))
print("statuses present:", sorted({r["status"] for r in orders}))

## The metric, named before it is computed

Kalpa Retail cares about returns. An order with status `returned` came back, and a returned order costs the business the delivery, the reverse logistics and usually the margin.

So today's rate is the **return rate**, and it is the first thing in this programme where **lower is better**. Keep that in your head, because a ranking of return rates puts the best performer at the top of the list with the smallest number beside it.

```
return rate  =  orders with status "returned"  /  all orders in the group
```

Client zero's own metric list names this one as first computed in Week 1 Thursday and returning as the modelling target in Week 5. You will meet this exact number again.

## Where this is going, before we build any of it

```
segment        orders   median amount   return rate
Business            9        Rs 2,050          11.1%
Student            10        Rs 1,430          20.0%
Retail-Core        14        Rs 1,910          35.7%
Retail-Plus        11        Rs 1,435          36.4%
```

You will have built this inside the hour. Then you will spend the rest of the session refusing to send it in that form.

## Section 1: counting into named piles

Monday you counted orders that passed a test. One counter, one number out.

Today you need one counter per segment. The natural move is a dictionary with a counter behind each name.

```
   orders                        counts
   ------                       --------
   Student      ---------->     Student     : 1, 2, 3 ...
   Retail-Core  ---------->     Retail-Core : 1, 2 ...
   Retail-Core  ------|
   Business     ------|---->    Business    : 1 ...
```

Here is the version a working analyst writes from memory.

In [ ]:
def count_with_remembered_segments():
    counts = {"Retail-Core": 0, "Retail-Plus": 0, "Business": 0}
    for r in orders:
        counts[r["segment"]] = counts[r["segment"]] + 1
    return counts

show_failure(count_with_remembered_segments)

### The deliberate failure of this half

```
KeyError: 'Student'
```

Read it the way Tuesday taught you, from the bottom.

`KeyError` means a dictionary was handed a key it does not hold. Python then prints the key it did not hold, in quotes: `'Student'`.

The dictionary held three segments because a person typed three segments. Kalpa Retail has four.

**Every hard-coded list of categories is a promise about data you have not read yet.** It survives exactly as long as nobody adds a segment, and nobody sends an email when they do.

Note where it failed: on the very first order in the file, `KR4200`, which is a Student order. This one is loud. The dangerous version of this mistake is the one that runs for six months and breaks the week a new segment launches.

### The fix: build the key the first time you see it

In [ ]:
counts = {}

for r in orders:
    key = r["segment"]
    if key not in counts:
        counts[key] = 0
    counts[key] = counts[key] + 1

for key in sorted(counts):
    print("{:<12} {:>3}".format(key, counts[key]))
print("{:<12} {:>3}".format("total", sum(counts.values())))

Four segments, discovered by reading rather than by remembering. The total matches your profiled order count, and that last line is how you catch a counting bug without being told there was one.

The same shape written shorter, which you have already met:

```python
counts[key] = counts.get(key, 0) + 1
```

That is Monday's `.get()` with a default, doing exactly what the three-line version does. Both are correct. Use whichever your reader will understand faster.

One number here deserves a second look. Student holds **ten** orders. The raw file held twelve, and two Student orders were rejected yesterday for failing conversion. Your cleaning changed the denominator of your smallest segment by seventeen percent, and nothing in today's code would tell you that. Yesterday's rejects log is the only place it is written down.

## Section 2: three things per segment, in one pass

Per segment you want the count, a typical amount and the return rate.

Count and returns accumulate one order at a time. The median cannot: it needs every value in the group, sorted, before it can be taken. So the pass collects the amounts into a list and the median is computed after the loop finishes.

```
   one pass over orders
          |
          +--> count     += 1
          +--> returned  += 1 if status is returned
          +--> amounts   .append(amount)
          |
     after the loop
          |
          +--> median = middle of sorted(amounts)
          +--> rate   = returned / count
```

In [ ]:
def summarise_by(orders, key_field):
    # Group orders on key_field: count, median amount and return rate per group.
    buckets = {}
    for r in orders:
        key = r[key_field]
        if key not in buckets:
            buckets[key] = {"count": 0, "amounts": [], "returned": 0}
        buckets[key]["count"] += 1
        buckets[key]["amounts"].append(r["amount"])
        if r["status"] == "returned":
            buckets[key]["returned"] += 1

    summary = {}
    for key, b in buckets.items():
        summary[key] = {
            "count": b["count"],
            "median_amount": statistics.median(b["amounts"]),
            "returned": b["returned"],
            "rate": b["returned"] / b["count"],
        }
    return summary


by_segment = summarise_by(orders, "segment")

for key in sorted(by_segment):
    s = by_segment[key]
    print("{:<12} count {:>3}   median Rs {:>7,.0f}   returned {:>2}   rate {:>5.1f}%".format(
        key, s["count"], s["median_amount"], s["returned"], s["rate"] * 100))

Note the parameter. `summarise_by` takes the field to group on rather than hard-coding `"segment"`, which costs nothing today and is the only reason your take-home is a four-line job instead of a copy-paste job.

Note also what is absent from every row: the mean. Notebook 1 settled that, and the setting holds here.

## Section 3: the ranking, and what it is really measuring

Sort by return rate, best first. Lower is better, so best first means smallest first.

In [ ]:
ranked = sorted(by_segment.items(), key=lambda pair: pair[1]["rate"])

print("return rate, best first")
print("-" * 30)
for key, s in ranked:
    print("{:<12} {:>5.1f}%".format(key, s["rate"] * 100))

A clean ranking with a clear winner. Business returns at 11.1 percent, less than a third of the worst segment.

Somebody in a meeting is now about to move budget towards Business. Print one more column before they do.

In [ ]:
print("return rate, best first, with the count it rests on")
print("-" * 52)
for key, s in ranked:
    print("{:<12} {:>5.1f}%   on {:>2} orders".format(key, s["rate"] * 100, s["count"]))

### The trap, sprung

The two best-performing segments in the file are the two smallest segments in the file.

Business wins on nine orders. One order separates it from a completely different story.

In [ ]:
b = by_segment["Business"]
st_ = by_segment["Student"]

print("Business: {} returned of {}  ->  {:.1f}%".format(b["returned"], b["count"], b["rate"] * 100))
print("Student:  {} returned of {}  ->  {:.1f}%".format(st_["returned"], st_["count"], st_["rate"] * 100))
print()
for extra in range(0, 3):
    swung = (b["returned"] + extra) / b["count"]
    verdict = "still best" if swung < st_["rate"] else "no longer best"
    print("if {} more Business order(s) had been returned: {:>5.1f}%   {}".format(extra, swung * 100, verdict))

**One order.** One more Business return and the best segment in the file is no longer the best segment in the file.

One order out of forty-four, in a segment of nine, on a difference somebody was about to fund.

This is not a flaw in these particular orders. Small groups produce extreme rates in both directions, in any dataset, with nothing causing it. Rank a set of groups by a rate and the smallest groups drift towards both ends of the list, so the ranking reads group size as much as it reads performance.

The fix is not a cleverer statistic. It is the column you already printed.

## Section 4: the honest sentence

The deliverable of this session is not the table. A table gets screenshotted, cropped, and pasted into a slide by somebody who never saw your notebook.

The deliverable is a sentence per segment that survives that journey. Three parts, in this order:

```
[the number]   [the denominator]   [what it does not yet support]
```

The third part is the one that goes missing, so it goes in the sentence rather than in a footnote.

In [ ]:
TRUST_FLOOR = 30   # a threshold you should be prepared to defend, not a law of nature

for key, s in ranked:
    line = "{}: returned on {} of {} orders, {:.1f} percent.".format(
        key, s["returned"], s["count"], s["rate"] * 100)
    if s["count"] < TRUST_FLOOR:
        line += " Fewer than {} orders, so treat this as an indication rather than a measurement.".format(TRUST_FLOOR)
    print(line)
    print()

Every segment in this file trips the floor, which is itself the finding. A forty-four order file does not support four confident segment claims, and saying so is the correct professional answer rather than a failure to produce one.

Written out for a person, with the money column handled the way notebook 1 settled:

```
Business:    returned on 1 of 9 orders, 11.1 percent. The lowest return rate in the file
             and the smallest segment in it. One more return takes it to 22.2 percent and
             behind Student, so this is not yet a finding.

Student:     returned on 2 of 10 orders, 20.0 percent. Ten orders, and the raw file held
             twelve before two failed conversion. Treat as unmeasured.

Retail-Core: returned on 5 of 14 orders, 35.7 percent. The largest segment in the file and
             the steadiest number here. Median order Rs 1,910; the mean of Rs 36,027.14 is
             the Rs 480,000 order and should not be quoted.

Retail-Plus: returned on 4 of 11 orders, 36.4 percent. Highest return rate in the file, and
             within one order of Retail-Core. The two are not separated by this data.
```

Two of the four decline to make a claim and a third says two segments cannot be told apart. That is the sentence doing its job.

In [ ]:
core = [r["amount"] for r in orders if r["segment"] == "Retail-Core"]
print("Retail-Core, the segment holding the whale:")
print("  orders:", len(core))
print("  median: Rs {:>9,.2f}".format(statistics.median(core)))
print("  mean:   Rs {:>9,.2f}   <- one order".format(sum(core) / len(core)))

### Milestone: what you can now answer

**Where this shows up in production.** Any dashboard that ranks stores, regions, cohorts or campaigns by a conversion or return rate has this built in, and the smallest units sit at the top and the bottom of the leaderboard month after month while the middle stays still. Teams that have been bitten once report the denominator beside every rate and grey out any row below a stated minimum. That minimum is a judgement somebody wrote down and defended, and being the person who writes it down is the job.

**Interview question this section just made answerable.**

> Segment A converts at 42 percent on 12 records and segment B at 31 percent on 1,200. Which do you trust?

Trust segment B's number. Say why in terms of movement: one record flips segment A's rate by more than eight points, while one record moves segment B by less than a tenth of a point. Then add the part most candidates leave out, which is that segment A is not reported as bad or good, it is reported as unmeasured, and you would say what volume it needs before the number means anything.

You have a better version of this answer than most candidates, because you can say it about your own file: Business leads on nine orders and one order takes the lead away.

## The question this session does not answer

Is Business genuinely better than Retail-Plus, or is 11.1 against 36.4 what four groups of these sizes do on their own?

That question has a real answer and a method behind it. It is Monday's session.

Today you name it, write it down, and stop. Write this line into your own notebook now, because you will open it on Monday:

> Open question: Business 11.1 percent on 9 orders against Retail-Plus 36.4 percent on 11 orders. Is that gap real? Method needed to settle it.

Saying "we do not know yet, and here is exactly what would tell us" is a complete answer in a room full of people who wanted a different one.

## What this notebook settled

| Question | Answer |
|---|---|
| How do you group without knowing the categories | Build the key on first sight; a remembered list gives you `KeyError: 'Student'` |
| Which statistic per segment | Median amount, because the money column has a tail |
| What goes beside every rate | The count it rests on, on the same line |
| Which segments lead on return rate | The two smallest, Business on 9 and Student on 10 |
| How much evidence is behind the lead | One order |
| Is the gap real | Unanswered today by design, and named as Monday's work |

**Crux.** Every rate carries its denominator, or it lies for you while you are not in the room.

Week 2 re-expresses this exact pass as one line of pandas and one SQL `GROUP BY` on the same orders. You built it by hand once so that when the one-liner arrives you already know what the answer should be, and you will notice if it disagrees.